# 롱 컨텍스트 처리 - 실습 코드 2: Needle-in-a-Haystack 테스트 구현

- Tutorial ID: `expand-long-context`
- Tutorial: 롱 컨텍스트 처리
- Section ID: `expand-long-context-code-2`
- Section: 실습 코드 2: Needle-in-a-Haystack 테스트 구현


In [ ]:
# ============================================================
# 코드 읽는 법 — 실습 코드 2: Needle-in-a-Haystack 테스트 구현
#
# 이 코드는 “정답을 한 번 실행”하는 용도가 아니라,
# 수학/아키텍처 개념이 실제 배열·텐서 연산으로 바뀌는 과정을
# 한 줄씩 추적하기 위한 실험 노트입니다.
#
# 학습 목표:
#   1) Q/K/V가 어떤 shape으로 만들어지고 attention score로 이어지는지 추적
#   2) 위치 정보가 벡터 회전/편향으로 attention score에 들어가는 방식 확인
#
# 읽는 순서:
#   1) 차원/하이퍼파라미터(batch_size, seq_len, d_model 등)를 먼저 확인합니다.
#   2) 입력 배열 X 또는 토큰/문서 데이터가 어떻게 만들어지는지 봅니다.
#   3) W_Q/W_K/W_V/W_O 같은 가중치 행렬이 어떤 공간으로 투영하는지 확인합니다.
#   4) @, matmul, softmax, mask, loss 등 핵심 연산 직후의 shape와 값을 출력으로 검증합니다.
#   5) seed, 차원, temperature, rank, top_k, expert 수 등을 바꿔 결과가 어떻게 변하는지 실험합니다.
#
# 주의:
#   - 숫자 하나하나를 외우기보다 “shape 변화”와 “정보가 이동하는 방향”을 보세요.
#   - torch/transformers/openai/vLLM 의존 코드는 Colab/로컬/서버 노트북 실행을 권장합니다.

In [ ]:
from openai import OpenAI
import random
import string
import time

client = OpenAI()

# ── 더미 컨텍스트 생성 ──
def generate_random_context(length_tokens=100000):
    """롱 컨텍스트 테스트용 더미 텍스트 생성"""
    # 실제로는 의미 있는 텍스트 사용 권장
    words = []
        for _ in range(length_tokens):
        word_len = random.randint(3, 10)
        word = ''.join(random.choices(string.ascii_lowercase, k=word_len))
        words.append(word)
    return ' '.join(words)

def insert_needle(haystack, needle, position_pct):
    """건초더미 중간에 바늘(핀) 삽입"""
    total_len = len(haystack)
    insert_pos = int(total_len * position_pct)
    return haystack[:insert_pos] + f"\n{needle}\n" + haystack[insert_pos:]

# ── Needle-in-a-Haystack 테스트 ──
def needle_in_haystack_test(
    model="gpt-4o",
    context_lengths=[1000, 2000, 4000, 8000, 16000, 32000],
    depth_percentages=[0, 10, 20, 30, 50, 70, 90, 100],
    needle="The secret code for the vault is 7294.",
    question="What is the secret code for the vault?",
):
    """다양한 컨텍스트 길이와 깊이에서 검색 능력 테스트"""
    results = []
    
        for ctx_len in context_lengths:
                for depth_pct in depth_percentages:
            # 건초더미 생성 + 바늘 삽입
            haystack = generate_random_context(ctx_len)
            full_context = insert_needle(haystack, needle, depth_pct / 100)
            
            # 토큰 수 추정 (영어 기준 ~4자/토큰)
            approx_tokens = len(full_context) // 4
            
            try:
                start = time.time()
                response = client.chat.completions.create(
                    model=model,
                    messages=[
                        {"role": "system", "content": "You must find specific information in the provided text."},
                        {"role": "user", "content": f"Context:\n{full_context}\n\nQuestion: {question}"}
                    ],
                    max_tokens=100,
                                        temperature=0,
                )
                answer = response.choices[0].message.content
                elapsed = time.time() - start
                
                # 정답 포함 여부
                correct = "7294" in answer
                
                results.append({
                    "context_tokens": approx_tokens,
                    "depth_pct": depth_pct,
                    "correct": correct,
                    "answer": answer[:100],
                    "time": elapsed,
                })
                
                status = "✅" if correct else "❌"
                print(f"  {status} ctx={approx_tokens:>6}토큰, depth={depth_pct:>3}% → {answer[:50]}")
                
            except Exception as e:
                print(f"  ⚠️ ctx={approx_tokens}, depth={depth_pct}% → Error: {str(e)[:50]}")
                results.append({
                    "context_tokens": approx_tokens,
                    "depth_pct": depth_pct,
                    "correct": False,
                    "answer": f"Error: {str(e)[:50]}",
                    "time": 0,
                })
    
    # 결과 요약
    total = len(results)
    correct = sum(1 for r in results if r['correct'])
    print(f"\n=== 결과 요약 ===")
    print(f"전체 정확도: {correct}/{total} ({correct/total*100:.1f}%)")
    
    # 깊이별 정확도
        for depth in depth_percentages:
        depth_results = [r for r in results if r['depth_pct'] == depth]
        d_correct = sum(1 for r in depth_results if r['correct'])
        print(f"  depth={depth:>3}%: {d_correct}/{len(depth_results)} ({d_correct/len(depth_results)*100:.0f}%)")
    
    return results

# 테스트 실행
print("=== Needle-in-a-Haystack 테스트 ===\n")
results = needle_in_haystack_test(
    context_lengths=[2000, 8000, 16000],
    depth_percentages=[0, 25, 50, 75, 100],
)